In [ ]:
import os
import sys
import oracledb

# Initialize Oracle Client in Thick mode for EBS compatibility
INSTANT_CLIENT_DIR = r"C:\Users\E1445612\oracle64client"

try:
    oracledb.init_oracle_client(lib_dir=INSTANT_CLIENT_DIR)
    print(f"[+] Oracle Client initialized in Thick mode from: {INSTANT_CLIENT_DIR}")
except Exception as e:
    print(f"[-] Failed to initialize Oracle Client: {e}")
    sys.exit(1)

# Database connection credentials
DB_USER = "apps"
DB_PASS = "Sukh#na7"
DB_DSN = "BU2EBS.EMRSN.COM:34101/BETSYU2"


def fetch_ap_invoices():
    print("\n" + "=" * 70)
    print(" FETCHING FIRST 5 ROWS FROM AP_INVOICES_ALL")
    print("=" * 70)

    print(f"[+] Connecting to Oracle EBS Database...")
    try:
        connection = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=DB_DSN)
        cursor = connection.cursor()
        print("[+] Connected successfully to EBS Instance.\n")
    except Exception as exc:
        print(f"[-] Database Connection Failed: {exc}")
        return

    # Query key columns using ROWNUM for Oracle EBS database compatibility
    query = """
    SELECT invoice_id,
           invoice_num,
           vendor_id,
           invoice_amount,
           invoice_date,
           invoice_type_lookup_code
      FROM apps.ap_invoices_all
     WHERE ROWNUM <= 5
    """

    try:
        print("[+] Executing query...")
        cursor.execute(query)

        # Get column headers dynamically from cursor description
        columns = [col[0] for col in cursor.description]
        rows = cursor.fetchall()

        if not rows:
            print("[-] No records found in AP_INVOICES_ALL.")
        else:
            print(f"[+] Retrieved {len(rows)} record(s):\n")
            print("-" * 110)
            print(
                f"{columns[0]:<12} | {columns[1]:<25} | {columns[2]:<10} | {columns[3]:<12} | {columns[4]:<12} | {columns[5]}"
            )
            print("-" * 110)

            for row in rows:
                inv_id, inv_num, v_id, amt, dt, typ = row
                dt_str = dt.strftime('%d-%b-%Y').upper() if dt else 'N/A'
                print(
                    f"{inv_id:<12} | {str(inv_num):<25} | {str(v_id):<10} | {str(amt):<12} | {dt_str:<12} | {typ}"
                )

            print("-" * 110)

    except Exception as exc:
        print(f"[-] Query Execution Error: {exc}")
    finally:
        cursor.close()
        connection.close()
        print("\n[+] Database connection closed.")


if __name__ == "__main__":
    fetch_ap_invoices()